# ¿Qué direcciones conserva un rostro transformado?
## Transformaciones lineales y valores propios mediante experimentación visual

**Actividad computacional - Álgebra Lineal básico** | **Modalidad:** Parejas | **Tiempo máximo:** 2 horas

**Integrante A:** _____________________________  **Integrante B:** _____________________________  
**Fecha:** _______________

---
### Propósito

Trabajarán con fotografías reales del conjunto **Olivetti Faces**. Sobre un mismo rostro se dibujarán vectores de coordenadas y luego se aplicarán transformaciones lineales $2\times2$. La información generada al ejecutar las celdas les permitirá:

- observar cómo cambia una imagen bajo una transformación lineal;
- buscar direcciones que se conservan;
- formular la idea de valor y vector propio;
- verificar sus conjeturas mediante productos matriz-vector.

Solo se utilizarán matrices cuyos valores propios son reales.

### Secuencia y tiempos

| Etapa | Actividad | Tiempo |
|---|---|---:|
| Observar | Fotografías reales y sistema de coordenadas | 10 min |
| Experimentar I | Escala y reflexión: buscar direcciones conservadas | 25 min |
| Formalizar | Definir y verificar valores/vectores propios | 25 min |
| Experimentar II | Cizalla: contrastar una conjetura | 20 min |
| Transferir | Reto con una nueva matriz y un rostro real | 20 min |
| Cerrar | Síntesis y entrega | 10 min |
| **Total** | | **110 min** |

### Trabajo en pareja

1. Alternen quién ejecuta y quién registra los resultados en cada experimento.
2. En las celdas marcadas **EXPLORACIÓN**, pueden modificar únicamente la lista `vectores_a_probar`.
3. En la celda marcada **VERIFICACIÓN**, ingresen únicamente su matriz, su vector candidato y su factor candidato.
4. No utilicen funciones automáticas para calcular valores propios: la evidencia será la imagen, las flechas y los productos $A\mathbf v$.


In [ ]:
# CONFIGURACIÓN - Ejecuten esta celda primero
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import affine_transform
from sklearn.datasets import fetch_olivetti_faces
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["figure.dpi"] = 100

print("Librerías cargadas correctamente.")


---
## 1. Observar: fotografías y coordenadas (10 minutos)

Cada foto tiene $64\times64$ píxeles. Una imagen puede guardarse como un vector de $4096$ intensidades, pero en esta actividad transformaremos las **posiciones** alrededor del centro del rostro:

$$\mathbf p=\begin{pmatrix}\text{fila}\\\text{columna}\end{pmatrix},
\qquad T(\mathbf p)=A\mathbf p.$$

La flecha azul que aparecerá en el rostro original representa $\mathbf v$. La flecha naranja sobre el rostro transformado representa $A\mathbf v$.


In [ ]:
# DATOS REALES Y HERRAMIENTAS DE EXPLORACIÓN
print("Cargando Olivetti Faces; la primera descarga puede tomar unos segundos...")
dataset = fetch_olivetti_faces(shuffle=True, random_state=42)
images = dataset.images
faces = dataset.data
labels = dataset.target

rostro_ref = images[np.where(labels == 5)[0][0]].copy()
centro = (np.array(rostro_ref.shape) - 1) / 2.0

fig, axes = plt.subplots(1, 4, figsize=(9, 2.4))
for ax, pid in zip(axes, [0, 5, 12, 33]):
    foto = images[np.where(labels == pid)[0][0]]
    ax.imshow(foto, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Persona {pid}")
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Forma de una fotografía: {rostro_ref.shape}; como vector: {rostro_ref.reshape(-1).shape}")

def transformar_rostro(imagen, A):
    # scipy usa la transformación inversa para ubicar los píxeles de salida.
    A_inv = np.linalg.inv(A)
    offset = centro - A_inv @ centro
    return affine_transform(imagen, A_inv, offset=offset, mode="constant", cval=0.0)

def dibujar_vector(ax, v, color, etiqueta, escala=12):
    # El primer componente es fila (vertical) y el segundo, columna (horizontal).
    ax.arrow(centro[1], centro[0], escala * v[1], escala * v[0],
             color=color, linewidth=2, head_width=2.0,
             length_includes_head=True)
    ax.text(centro[1] + escala * v[1], centro[0] + escala * v[0],
            etiqueta, color=color, fontsize=10, fontweight="bold")

def explorar_transformacion(A, nombre, vectores):
    imagen_t = transformar_rostro(rostro_ref, A)
    fig, axes = plt.subplots(len(vectores), 2, figsize=(7, 2.5 * len(vectores)))
    if len(vectores) == 1:
        axes = np.array([axes])

    print(f"Transformación: {nombre}")
    print(f"Matriz A =\n{A}\n")
    print(f"{'v':>14} | {'A v':>16} | ¿misma línea?")
    print("-" * 51)
    for fila, v in enumerate(vectores):
        v = np.array(v, dtype=float)
        Av = A @ v
        misma_linea = abs(v[0] * Av[1] - v[1] * Av[0]) < 1e-9
        texto = "sí" if misma_linea else "no"
        print(f"{str(v):>14} | {str(np.round(Av, 3)):>16} | {texto}")

        axes[fila, 0].imshow(rostro_ref, cmap="gray", vmin=0, vmax=1)
        dibujar_vector(axes[fila, 0], v, "royalblue", "v")
        axes[fila, 0].set_title(f"Original: v={v.astype(int)}")

        axes[fila, 1].imshow(imagen_t, cmap="gray", vmin=0, vmax=1)
        dibujar_vector(axes[fila, 1], Av, "darkorange", "Av")
        axes[fila, 1].set_title(f"Transformado: Av={np.round(Av, 2)}")

        for ax in axes[fila]:
            ax.axis("off")
    plt.tight_layout()
    plt.show()


### ✏️ Registro 1

**R1.1** ¿Cuántas entradas tiene el vector que almacena una fotografía? ¿Qué representa cada entrada?

> **Respuesta:**

**R1.2** En las visualizaciones que siguen, ¿qué representa una flecha sobre el rostro y por qué una matriz $2\times2$ puede actuar sobre ella aunque la imagen tenga 4096 intensidades?

> **Respuesta:**


---
## 2. Experimentar I: escala y reflexión (25 minutos)

En una dirección especial, la flecha transformada puede quedar sobre la **misma línea** que la flecha original, aunque cambie su tamaño o sentido.

Ejecuten cada experimento con los tres vectores propuestos. Después cambien uno de los vectores por otro que elijan ustedes, por ejemplo `(2, 1)` o `(1, -2)`, y vuelvan a ejecutar.


In [ ]:
# EXPLORACIÓN A - ESCALA
A_escala = np.array([[1.30, 0.00],
                     [0.00, 0.70]])

# Modifiquen únicamente esta lista para probar un vector adicional.
vectores_a_probar = [(1, 0), (0, 1), (1, 1)]
explorar_transformacion(A_escala, "Escala A_E", vectores_a_probar)


In [ ]:
# EXPLORACIÓN B - REFLEXIÓN IZQUIERDA/DERECHA
A_reflexion = np.array([[1.00,  0.00],
                        [0.00, -1.00]])

# Modifiquen únicamente esta lista para probar un vector adicional.
vectores_a_probar = [(1, 0), (0, 1), (1, 1)]
explorar_transformacion(A_reflexion, "Reflexión A_R", vectores_a_probar)


### ✏️ Registro 2: observación y conjetura

Completen la tabla usando las flechas y la tabla impresa por las celdas.

| Matriz | Vector probado $\mathbf v$ | Resultado $A\mathbf v$ | ¿Queda en la misma línea? | Si sí, factor observado |
|---|---|---|:---:|:---:|
| $A_E$ | $(1,0)$ | | | |
| $A_E$ | $(0,1)$ | | | |
| $A_E$ | vector elegido: ______ | | | |
| $A_R$ | $(1,0)$ | | | |
| $A_R$ | $(0,1)$ | | | |
| $A_R$ | vector elegido: ______ | | | |

**R2.1** En sus palabras, ¿qué característica común tienen los vectores que quedaron en la misma línea después de aplicar la matriz?

> **Conjetura de la pareja:**

**R2.2** En la reflexión, una flecha puede quedar sobre la misma línea pero apuntar al sentido opuesto. ¿Qué signo debería tener el factor que la relaciona con su imagen transformada?

> **Respuesta basada en la exploración:**


---
## 3. Formalizar y verificar: valores y vectores propios (25 minutos)

La observación anterior se formaliza así:

$$A\mathbf v=\lambda\mathbf v, \qquad \mathbf v\ne\mathbf 0.$$

Si la igualdad se cumple, $\mathbf v$ es un **vector propio** y $\lambda$ es su **valor propio**. El número $\lambda$ describe el cambio sobre esa dirección:

| Valor de $\lambda$ | Interpretación visible |
|---|---|
| $\lambda>1$ | la flecha se alarga sin invertir sentido |
| $0<\lambda<1$ | la flecha se acorta sin invertir sentido |
| $\lambda=1$ | la flecha queda igual |
| $\lambda<0$ | la flecha invierte sentido |

Utilicen primero la evidencia obtenida, y luego comprueben mediante multiplicación.


In [ ]:
# VERIFICACIÓN DE UNA CONJETURA - Esta celda no calcula respuestas.
# El ejemplo inicial es un intento que NO verifica. Sustitúyanlo por su conjetura.
A = A_escala                 # Cambien por A_reflexion, A_cizalla o A_desafio si corresponde.
v_candidato = np.array([1.0, 1.0])
lambda_candidato = 1.00

Av = A @ v_candidato
lambda_v = lambda_candidato * v_candidato
error = np.linalg.norm(Av - lambda_v)

print(f"A v       = {np.round(Av, 4)}")
print(f"lambda v  = {np.round(lambda_v, 4)}")
print(f"Error      = {error:.6f}")
print("¿Verifica A v = lambda v?", "SÍ" if error < 1e-8 else "NO; revisen su conjetura")


### ✏️ Registro 3: formalización

**R3.1** Elijan dos pares observados en la escala, uno por cada dirección conservada. Escriban la igualdad $A_E\mathbf v=\lambda\mathbf v$ y verifíquenla con la celda anterior, cambiando los datos de entrada.

| $\mathbf v$ | $\lambda$ conjeturado | $A_E\mathbf v$ | $\lambda\mathbf v$ | ¿Verifica? |
|---|:---:|---|---|:---:|
| | | | | |
| | | | | |

**R3.2** Hagan lo mismo con las dos direcciones conservadas que encontraron para la reflexión.

| $\mathbf v$ | $\lambda$ conjeturado | $A_R\mathbf v$ | $\lambda\mathbf v$ | ¿Qué se observa en el rostro? |
|---|:---:|---|---|---|
| | | | | |
| | | | | |

**R3.3** Sin usar una función automática, calculen $\det(A_R-\lambda I)=0$. ¿Coinciden los valores obtenidos con los factores descubiertos experimentalmente?

> **Cálculo y conclusión:**


---
## 4. Experimentar II: una cizalla (20 minutos)

Una transformación puede conservar alguna dirección sin conservarlas todas. Investiguen la cizalla:

$$A_C=\begin{pmatrix}1&0.45\\0&1\end{pmatrix}.$$

Prueben primero los vectores sugeridos; después modifiquen la lista para ensayar al menos dos vectores propios de la pareja.


In [ ]:
# EXPLORACIÓN C - CIZALLA
A_cizalla = np.array([[1.00, 0.45],
                      [0.00, 1.00]])

# Modifiquen únicamente esta lista; documenten por lo menos cinco intentos en total.
vectores_a_probar = [(1, 0), (0, 1), (1, 1)]
explorar_transformacion(A_cizalla, "Cizalla A_C", vectores_a_probar)


### ✏️ Registro 4: contrastar y explicar

| Vector probado | $A_C\mathbf v$ observado | ¿Misma línea? | Factor si existe |
|---|---|:---:|:---:|
| $(1,0)$ | | | |
| $(0,1)$ | | | |
| $(1,1)$ | | | |
| elección 1: ______ | | | |
| elección 2: ______ | | | |

**R4.1** Formulen una conjetura sobre cuáles vectores son propios de $A_C$ y qué valor propio tienen. Verifiquen al menos uno en la celda **VERIFICACIÓN**.

> **Conjetura y verificación:**

**R4.2** La cizalla tiene una dirección que queda igual, pero la imagen transformada no es igual a la original. Expliquen esta aparente contradicción usando sus experimentos con otras flechas.

> **Respuesta:**


---
## 5. Transferir: nueva transformación (20 minutos)

Ahora trabajarán con una matriz nueva:

$$A_D=\begin{pmatrix}1.20&0.30\\0&0.60\end{pmatrix}.$$

La celda muestra la imagen transformada y permite probar vectores, pero **no les dice** cuáles son los vectores propios. Su tarea es encontrarlos experimentalmente, formular valores candidatos y verificarlos.


In [ ]:
# EXPLORACIÓN D - RETO EN PAREJA
A_desafio = np.array([[1.20, 0.30],
                      [0.00, 0.60]])

# Empiecen con estos tres. Después prueben al menos tres vectores distintos elegidos por ustedes.
vectores_a_probar = [(1, 0), (0, 1), (1, 1)]
explorar_transformacion(A_desafio, "Reto A_D", vectores_a_probar)


### ✏️ Registro 5: descubrimiento en pareja

**R5.1** Registren por lo menos seis intentos. Busquen dos direcciones que permanezcan sobre la misma línea.

| Vector probado | $A_D\mathbf v$ | ¿Misma línea? | Factor candidato |
|---|---|:---:|:---:|
| $(1,0)$ | | | |
| $(0,1)$ | | | |
| $(1,1)$ | | | |
| elección 1: ______ | | | |
| elección 2: ______ | | | |
| elección 3: ______ | | | |

**R5.2** Escriban sus dos conjeturas en la forma $A_D\mathbf v=\lambda\mathbf v$ y compruébenlas usando la celda **VERIFICACIÓN**.

> **Conjeturas verificadas:**

**R5.3** Ahora calculen $\det(A_D-\lambda I)=0$ en papel. Comparen los valores obtenidos algebraicamente con los factores que encontraron experimentando.

> **Cálculo y comparación:**

**R5.4** Describan qué parte del cambio del rostro se puede anticipar al conocer esas dos direcciones propias y qué rasgos visuales no se describen por una sola flecha.

> **Explicación conjunta:**


---
## Cierre y entrega (10 minutos)

### ✏️ Síntesis

En máximo 100 palabras, completen la secuencia:

`observamos flechas -> conjeturamos direcciones especiales -> definimos vector propio -> verificamos -> aplicamos a una nueva imagen transformada`.

Incluyan un ejemplo concreto de $(A,\mathbf v,\lambda)$ descubierto durante la actividad y su interpretación visible.

> **Respuesta conjunta:**

### Entregables

- Registros 1 a 5 completados.
- Al menos cuatro verificaciones propias ejecutadas y documentadas.
- Síntesis con nombres de ambos integrantes.


---
## Rúbrica de evaluación

| Evidencia | Puntos |
|---|:---:|
| R1: interpretación de la imagen, sus coordenadas y vectores | 10 |
| R2: observaciones y conjeturas en escala/reflexión | 20 |
| R3: formalización y verificación de $A\mathbf v=\lambda\mathbf v$ | 25 |
| R4: experimentación y explicación de la cizalla | 15 |
| R5: exploración, verificación y cálculo del reto | 25 |
| Síntesis final | 5 |
| **Total** | **100** |

**Se evalúa especialmente:** uso de la evidencia producida por las celdas, calidad de las conjeturas, comprobaciones registradas, cálculos algebraicos posteriores y participación de la pareja.

*Actividad diseñada para Álgebra Lineal | Fotografías: Olivetti Faces, AT&T Laboratories Cambridge (1992-1994).*
